# Jacobian Heatmap: Gene-Gene Regulatory Map

Compute and visualize the Jacobian matrix as a clustered heatmap.
Shows which input genes regulate which output genes under perturbation.

- **Rows**: Top 100 experimentally affected output genes
- **Columns**: Top 50 input genes by gradient importance
- **Color**: Blue = no influence, Red = strong influence
- **Targets**: GFI1B and TFRC

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install h5py tqdm scipy pandas -q

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from tqdm import tqdm
from scipy.stats import pearsonr
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import pdist
from matplotlib.colors import LinearSegmentedColormap
from dataclasses import dataclass
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'Arial'
matplotlib.rcParams['font.size'] = 10
import logging
logging.getLogger('matplotlib').setLevel(logging.ERROR)

print('Imports OK')

In [ ]:
# Paths
DRIVE_BASE = Path('/content/drive/MyDrive/cdt_data')
MODEL_BASE = Path('/content/drive/MyDrive/cdt_outputs/morris_crispri_stage1_5')
OUTPUT_BASE = Path('/content/drive/MyDrive/cdt_outputs/gradient_validation')
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

TSS_EFFECTS_PATH = DRIVE_BASE / 'morris_celllevel_effects_2361.h5'
TSS_ENFORMER_PATH = DRIVE_BASE / 'morris_28genes_enformer.h5'
MODEL_PATH = MODEL_BASE / 'cdt_morris_celllevel_best.pt'
GENE_LIST_PATH = DRIVE_BASE / 'k562_gene_embeddings_aligned.h5'

for name, path in [('Effects', TSS_EFFECTS_PATH), ('Enformer', TSS_ENFORMER_PATH),
                    ('Model', MODEL_PATH), ('Gene list', GENE_LIST_PATH)]:
    print(f'  [{"OK" if path.exists() else "NOT FOUND"}] {name}')

## Load Data

In [ ]:
# Load cell-level effects
with h5py.File(TSS_EFFECTS_PATH, 'r') as f:
    tss_log2fc = f['log2fc'][:]
    tss_cell_expr = f['cell_expr'][:]
    tss_target_gene_idx = f['target_gene_idx'][:]
    tss_target_gene_names = [g.decode() if isinstance(g, bytes) else g for g in f['target_gene_names'][:]]
    tss_val_genes = [g.decode() if isinstance(g, bytes) else g for g in f['val_genes'][:]]
    ntc_mean_expr = f['ntc_mean_expr'][:] if 'ntc_mean_expr' in f.keys() else None

N_GENES = tss_cell_expr.shape[1]
if ntc_mean_expr is None:
    ntc_mean_expr = tss_cell_expr.mean(axis=0)

# Load Enformer embeddings
with h5py.File(TSS_ENFORMER_PATH, 'r') as f:
    tss_enformer_emb = f['embeddings'][:]
    tss_enformer_genes = [g.decode() if isinstance(g, bytes) else g for g in f['gene_names'][:]]
tss_gene_to_enformer = {gene: i for i, gene in enumerate(tss_enformer_genes)}

# Load gene names
with h5py.File(GENE_LIST_PATH, 'r') as f:
    base_genes = [g.decode() if isinstance(g, bytes) else g for g in f['gene_names'][:]]
cdt_genes = base_genes + ['GFI1B'] if 'GFI1B' not in base_genes else base_genes
gene_to_idx = {g: i for i, g in enumerate(cdt_genes)}

print(f'Genes: {N_GENES}, Cells: {tss_log2fc.shape[0]}')
print(f'Val genes: {tss_val_genes}')
print(f'Enformer genes: {len(tss_enformer_genes)}')
print(f'Target genes available: {tss_target_gene_names}')

## Model Definition & Loading

In [ ]:
@dataclass
class CDTCRISPRiConfig:
    dna_dim: int = 3072
    dna_seq_len: int = 896
    n_genes: int = 2361
    hidden_dim: int = 512
    nhead: int = 8
    dropout: float = 0.3
    dna_self_attn_layers: int = 2
    rna_self_attn_layers: int = 1


class RawExpressionEncoder(nn.Module):
    def __init__(self, n_genes, hidden_dim, dropout=0.1):
        super().__init__()
        self.n_genes = n_genes
        self.hidden_dim = hidden_dim
        self.gene_embedding = nn.Embedding(n_genes, hidden_dim)
        self.expr_projector = nn.Sequential(
            nn.Linear(1, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(), nn.Dropout(dropout))
        self.combine = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.LayerNorm(hidden_dim), nn.Dropout(dropout))

    def forward(self, expression):
        batch_size = expression.size(0)
        gene_ids = torch.arange(self.n_genes, device=expression.device)
        gene_emb = self.gene_embedding(gene_ids).unsqueeze(0).expand(batch_size, -1, -1)
        expr_emb = self.expr_projector(expression.unsqueeze(-1))
        return self.combine(torch.cat([gene_emb, expr_emb], dim=-1))


class SequenceProjector(nn.Module):
    def __init__(self, input_dim, output_dim, dropout=0.1):
        super().__init__()
        self.linear = nn.Linear(input_dim, output_dim)
        self.norm = nn.LayerNorm(output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.norm(self.linear(x)))


class FlashSelfAttentionBlock(nn.Module):
    def __init__(self, d_model, nhead=8, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.nhead = nhead
        self.head_dim = d_model // nhead
        self.dropout_p = dropout
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model), nn.Dropout(dropout))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, S, _ = x.shape
        Q = self.q_proj(x).view(B, S, self.nhead, self.head_dim).transpose(1, 2)
        K = self.k_proj(x).view(B, S, self.nhead, self.head_dim).transpose(1, 2)
        V = self.v_proj(x).view(B, S, self.nhead, self.head_dim).transpose(1, 2)
        attn_out = F.scaled_dot_product_attention(Q, K, V, dropout_p=self.dropout_p if self.training else 0.0)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, S, self.d_model)
        attn_out = self.out_proj(attn_out)
        x = self.norm1(x + self.dropout(attn_out))
        x = self.norm2(x + self.ffn(x))
        return x


class FlashCrossAttentionBlock(nn.Module):
    def __init__(self, d_model, nhead=8, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.nhead = nhead
        self.head_dim = d_model // nhead
        self.dropout_p = dropout
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model), nn.Dropout(dropout))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key_value):
        B, QL, _ = query.shape
        KL = key_value.shape[1]
        Q = self.q_proj(query).view(B, QL, self.nhead, self.head_dim).transpose(1, 2)
        K = self.k_proj(key_value).view(B, KL, self.nhead, self.head_dim).transpose(1, 2)
        V = self.v_proj(key_value).view(B, KL, self.nhead, self.head_dim).transpose(1, 2)
        attn_out = F.scaled_dot_product_attention(Q, K, V, dropout_p=self.dropout_p if self.training else 0.0)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, QL, self.d_model)
        attn_out = self.out_proj(attn_out)
        x = self.norm1(query + self.dropout(attn_out))
        x = self.norm2(x + self.ffn(x))
        return x


class VirtualCellEmbedderDNARNA(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.nhead = 4
        self.head_dim = d_model // self.nhead
        self.dna_query = nn.Parameter(torch.randn(1, 1, d_model))
        self.rna_query = nn.Parameter(torch.randn(1, 1, d_model))
        self.dna_q_proj = nn.Linear(d_model, d_model)
        self.dna_k_proj = nn.Linear(d_model, d_model)
        self.dna_v_proj = nn.Linear(d_model, d_model)
        self.dna_out_proj = nn.Linear(d_model, d_model)
        self.rna_q_proj = nn.Linear(d_model, d_model)
        self.rna_k_proj = nn.Linear(d_model, d_model)
        self.rna_v_proj = nn.Linear(d_model, d_model)
        self.rna_out_proj = nn.Linear(d_model, d_model)
        self.fusion = nn.Sequential(
            nn.Linear(d_model * 2, d_model * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model), nn.LayerNorm(d_model))

    def _attention_pool(self, query, key_value, q_proj, k_proj, v_proj, out_proj):
        B = key_value.size(0)
        S = key_value.size(1)
        query = query.expand(B, -1, -1)
        Q = q_proj(query).view(B, 1, self.nhead, self.head_dim).transpose(1, 2)
        K = k_proj(key_value).view(B, S, self.nhead, self.head_dim).transpose(1, 2)
        V = v_proj(key_value).view(B, S, self.nhead, self.head_dim).transpose(1, 2)
        attn_out = F.scaled_dot_product_attention(Q, K, V)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, 1, self.d_model)
        return out_proj(attn_out).squeeze(1)

    def forward(self, dna_encoded, rna_encoded):
        dna_pooled = self._attention_pool(self.dna_query, dna_encoded,
            self.dna_q_proj, self.dna_k_proj, self.dna_v_proj, self.dna_out_proj)
        rna_pooled = self._attention_pool(self.rna_query, rna_encoded,
            self.rna_q_proj, self.rna_k_proj, self.rna_v_proj, self.rna_out_proj)
        return self.fusion(torch.cat([dna_pooled, rna_pooled], dim=-1))


class CDTCRISPRiModel(nn.Module):
    def __init__(self, config=None):
        super().__init__()
        if config is None:
            config = CDTCRISPRiConfig()
        self.config = config
        self.dna_projector = SequenceProjector(config.dna_dim, config.hidden_dim, config.dropout)
        self.dna_self_attn_layers = nn.ModuleList([
            FlashSelfAttentionBlock(config.hidden_dim, config.nhead, config.dropout)
            for _ in range(config.dna_self_attn_layers)])
        self.rna_encoder = RawExpressionEncoder(config.n_genes, config.hidden_dim, config.dropout)
        self.rna_self_attn_layers = nn.ModuleList([
            FlashSelfAttentionBlock(config.hidden_dim, config.nhead, config.dropout)
            for _ in range(config.rna_self_attn_layers)])
        self.dna_to_rna = FlashCrossAttentionBlock(config.hidden_dim, config.nhead, config.dropout)
        self.vce = VirtualCellEmbedderDNARNA(config.hidden_dim, config.dropout)
        self.task_layer = nn.Sequential(
            nn.Linear(config.hidden_dim, config.hidden_dim), nn.GELU(), nn.Dropout(config.dropout),
            nn.Linear(config.hidden_dim, config.n_genes))

    def forward(self, dna_emb, rna_expr):
        dna = self.dna_projector(dna_emb)
        rna = self.rna_encoder(rna_expr)
        for layer in self.dna_self_attn_layers:
            dna = layer(dna)
        for layer in self.rna_self_attn_layers:
            rna = layer(rna)
        rna = self.dna_to_rna(query=rna, key_value=dna)
        cell_embedding = self.vce(dna, rna)
        return self.task_layer(cell_embedding)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CDTCRISPRiModel(CDTCRISPRiConfig()).to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=True))
model.eval()
print(f'Model loaded on {device}')

## Compute Jacobian & Generate Heatmap

Function that computes Jacobian and generates clustered heatmap for any target gene.

In [ ]:
def compute_jacobian(target_gene, model, tss_enformer_emb, tss_gene_to_enformer,
                     ntc_mean_expr, tss_log2fc, tss_target_gene_idx,
                     tss_target_gene_names, cdt_genes, device, n_top_outputs=100):
    """Compute Jacobian for a target gene: d(output)/d(input) at NTC baseline."""
    N = len(cdt_genes)

    # Get experimental effects
    gene_idx_in_target = tss_target_gene_names.index(target_gene)
    cell_mask = tss_target_gene_idx == gene_idx_in_target
    mean_log2fc = tss_log2fc[cell_mask].mean(axis=0)
    exp_abs = np.abs(mean_log2fc)

    # Top affected output genes
    top_affected = np.argsort(exp_abs)[-n_top_outputs:][::-1]

    # DNA embedding for this gene
    dna = tss_enformer_emb[tss_gene_to_enformer[target_gene]]
    dna_t = torch.from_numpy(dna.astype(np.float32)).unsqueeze(0).to(device)
    rna_t = torch.from_numpy(ntc_mean_expr.astype(np.float32)).unsqueeze(0).to(device)
    rna_t.requires_grad_(True)

    # Compute Jacobian row by row
    jacobian = np.zeros((n_top_outputs, N), dtype=np.float32)
    for i, out_idx in enumerate(tqdm(top_affected, desc=f'{target_gene} Jacobian')):
        model.zero_grad()
        rna_t.grad = None
        pred = model(dna_t, rna_t)
        pred[0, out_idx].backward(retain_graph=True)
        jacobian[i] = rna_t.grad[0].cpu().numpy()

    mean_abs_grad = np.abs(jacobian).mean(axis=0)
    r, p = pearsonr(mean_abs_grad, exp_abs)
    print(f'{target_gene}: Pearson r = {r:.4f} (p = {p:.2e}), cells = {cell_mask.sum()}')

    return jacobian, top_affected, mean_abs_grad, exp_abs, r


def plot_jacobian_heatmap(jacobian, top_affected, mean_abs_grad, cdt_genes,
                          tss_val_genes, tss_target_gene_names, target_gene,
                          output_path, n_input_genes=50):
    """Generate clustered heatmap from Jacobian matrix."""
    abs_jac = np.abs(jacobian)

    # Select top input genes
    top_input_idx = np.argsort(mean_abs_grad)[-n_input_genes:][::-1]
    top_input_names = [cdt_genes[i] for i in top_input_idx]
    top_output_names = [cdt_genes[i] for i in top_affected]

    # Submatrix
    heatmap_data = abs_jac[:, top_input_idx]
    heatmap_log = np.log10(heatmap_data + 1e-8)

    # Hierarchical clustering
    row_order = leaves_list(linkage(pdist(heatmap_log, metric='correlation'), method='ward'))
    col_order = leaves_list(linkage(pdist(heatmap_log.T, metric='correlation'), method='ward'))

    heatmap_clustered = heatmap_log[row_order][:, col_order]
    row_labels = [top_output_names[i] for i in row_order]
    col_labels = [top_input_names[i] for i in col_order]

    # Colormap: blue (no influence) -> white -> red (strong influence)
    # log10 values: more negative = smaller gradient = blue, less negative = larger gradient = red
    cmap = LinearSegmentedColormap.from_list(
        'blue_white_red',
        ['#2166AC', '#67A9CF', '#D1E5F0', '#FDDBC7', '#EF8A62', '#B2182B'], N=256)

    val_set = set(tss_val_genes)
    target_set = set(tss_target_gene_names)

    fig, ax = plt.subplots(figsize=(16, 20))
    im = ax.imshow(heatmap_clustered, aspect='auto', cmap=cmap, interpolation='nearest')

    cbar = plt.colorbar(im, ax=ax, shrink=0.5, pad=0.02)
    cbar.set_label('log$_{10}$|$\\partial$output/$\\partial$input|', fontsize=11)

    ax.set_yticks(range(len(row_labels)))
    ax.set_yticklabels(row_labels, fontsize=5.5)
    ax.set_xticks(range(len(col_labels)))
    ax.set_xticklabels(col_labels, fontsize=7, rotation=90)

    # Color-code labels
    for i, label in enumerate(col_labels):
        if label in val_set:
            ax.get_xticklabels()[i].set_color('#E74C3C')
            ax.get_xticklabels()[i].set_fontweight('bold')
        elif label in target_set:
            ax.get_xticklabels()[i].set_color('#2980B9')
            ax.get_xticklabels()[i].set_fontweight('bold')

    for i, label in enumerate(row_labels):
        if label in val_set:
            ax.get_yticklabels()[i].set_color('#E74C3C')
            ax.get_yticklabels()[i].set_fontweight('bold')
        elif label in target_set:
            ax.get_yticklabels()[i].set_color('#2980B9')
            ax.get_yticklabels()[i].set_fontweight('bold')

    ax.set_xlabel('Input genes (top 50 by gradient importance)', fontsize=12)
    ax.set_ylabel('Output genes (top 100 experimentally affected)', fontsize=12)
    ax.set_title(f'Jacobian Regulatory Map: {target_gene} Perturbation\n'
                 f'|$\\partial$(output gene) / $\\partial$(input gene)| at NTC baseline',
                 fontsize=14)

    plt.tight_layout()
    plt.savefig(output_path.with_suffix('.pdf'), bbox_inches='tight')
    plt.savefig(output_path.with_suffix('.png'), dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Saved: {output_path.stem}.pdf/png')
    print(f'Red = strong regulatory influence, Blue = no influence')

print('Functions defined')

## GFI1B Jacobian Heatmap

In [ ]:
jac_gfi1b, top_out_gfi1b, grad_gfi1b, exp_gfi1b, r_gfi1b = compute_jacobian(
    'GFI1B', model, tss_enformer_emb, tss_gene_to_enformer,
    ntc_mean_expr, tss_log2fc, tss_target_gene_idx,
    tss_target_gene_names, cdt_genes, device)

plot_jacobian_heatmap(
    jac_gfi1b, top_out_gfi1b, grad_gfi1b, cdt_genes,
    tss_val_genes, tss_target_gene_names, 'GFI1B',
    OUTPUT_BASE / 'jacobian_heatmap_GFI1B')

## TFRC Jacobian Heatmap

In [ ]:
# Check TFRC availability
print(f'TFRC in Enformer: {"TFRC" in tss_gene_to_enformer}')
print(f'TFRC in targets: {"TFRC" in tss_target_gene_names}')

jac_tfrc, top_out_tfrc, grad_tfrc, exp_tfrc, r_tfrc = compute_jacobian(
    'TFRC', model, tss_enformer_emb, tss_gene_to_enformer,
    ntc_mean_expr, tss_log2fc, tss_target_gene_idx,
    tss_target_gene_names, cdt_genes, device)

plot_jacobian_heatmap(
    jac_tfrc, top_out_tfrc, grad_tfrc, cdt_genes,
    tss_val_genes, tss_target_gene_names, 'TFRC',
    OUTPUT_BASE / 'jacobian_heatmap_TFRC')

## TFRC Pathway Analysis: PPMX-T003 Side Effect Prediction

Cross-reference CDT-II's Jacobian-predicted downstream genes with known
biological pathways relevant to PPMX-T003 (anti-TfR1 antibody) clinical findings.

**PPMX-T003 reported side effects (Phase 1):**
- Reticulocyte decrease >50% (day 3)
- Mild anemia / microcytic anemia
- Hematocrit & hemoglobin decrease (day 7)
- Infusion-related reactions

**Question:** Do CDT-II's predicted downstream genes of TFRC perturbation
overlap with pathways that explain these clinical observations?

In [ ]:
# ══════════════════════════════════════════════════════════════════
# TFRC Pathway Analysis: Side Effect Prediction for PPMX-T003
# ══════════════════════════════════════════════════════════════════

# --- Curated pathway gene sets relevant to TFRC/iron biology ---
pathway_genes = {
    'Iron metabolism': [
        'FTH1', 'FTL', 'TFRC', 'SLC40A1', 'HAMP', 'HFE', 'TF', 'IREB2',
        'ACO1', 'SLC11A2', 'STEAP3', 'CYBRD1', 'SLC48A1', 'HMOX1', 'HMOX2',
        'ALAS1', 'ALAS2', 'FECH', 'ABCB7', 'MFRN1', 'MFRN2', 'FTMT',
        'PCBP1', 'PCBP2', 'NCOA4', 'CISD1', 'CISD2', 'BMP6',
    ],
    'Ferroptosis': [
        'GPX4', 'SLC7A11', 'SLC3A2', 'ACSL4', 'LPCAT3', 'ALOX15',
        'ALOX12', 'NFE2L2', 'KEAP1', 'FSP1', 'DHODH', 'GCH1',
        'CBS', 'CTH', 'GSS', 'GCLC', 'GCLM', 'GSR', 'TXNRD1',
        'HMOX1', 'NQO1', 'FTH1', 'FTL', 'NCOA4', 'ATG5', 'ATG7',
    ],
    'Erythropoiesis / Hematopoiesis': [
        'GATA1', 'GATA2', 'KLF1', 'TAL1', 'LMO2', 'EPOR', 'EPO',
        'GYPA', 'GYPB', 'GYPC', 'SLC4A1', 'ANK1', 'SPTA1', 'SPTB',
        'HBA1', 'HBA2', 'HBB', 'HBD', 'HBG1', 'HBG2', 'ALAS2',
        'FECH', 'ABCB10', 'TFRC', 'CD71', 'KIT', 'CKIT',
        'SPI1', 'RUNX1', 'MYB', 'GFI1', 'GFI1B', 'LDB1',
    ],
    'Cell proliferation / Survival': [
        'MYC', 'MYCN', 'CCND1', 'CCND2', 'CCNE1', 'CDK2', 'CDK4',
        'CDK6', 'RB1', 'TP53', 'MDM2', 'BCL2', 'MCL1', 'BAX', 'BAK1',
        'BIRC5', 'XIAP', 'PCNA', 'MKI67', 'TOP2A', 'E2F1',
    ],
    'Amino acid transport (LAT1 axis)': [
        'SLC7A5', 'SLC3A2', 'SLC1A5', 'SLC38A1', 'SLC38A2',
        'MTOR', 'RPTOR', 'RPS6KB1', 'EIF4EBP1', 'ATF4',
        'DDIT3', 'ASNS', 'SLC7A11', 'SLC7A1',
    ],
    'Oxidative stress response': [
        'NFE2L2', 'KEAP1', 'HMOX1', 'NQO1', 'GCLC', 'GCLM',
        'SOD1', 'SOD2', 'CAT', 'GPX1', 'GPX4', 'PRDX1', 'PRDX2',
        'TXN', 'TXNRD1', 'SRXN1', 'SQSTM1',
    ],
}

# --- Analyze TFRC Jacobian results ---
# Get top affected output genes and top input genes
tfrc_output_names = [cdt_genes[i] for i in top_out_tfrc]
top_input_idx = np.argsort(grad_tfrc)[-50:][::-1]
tfrc_input_names = [cdt_genes[i] for i in top_input_idx]
all_tfrc_genes = set(tfrc_output_names) | set(tfrc_input_names)

print('='*70)
print('TFRC PERTURBATION: PATHWAY ENRICHMENT ANALYSIS')
print('Relevance to PPMX-T003 (anti-TfR1 antibody) side effects')
print('='*70)

pathway_results = []

for pathway_name, pathway_gene_list in pathway_genes.items():
    # Check overlap with output genes (top 100 affected)
    output_overlap = set(tfrc_output_names) & set(pathway_gene_list)
    # Check overlap with input genes (top 50 by gradient)
    input_overlap = set(tfrc_input_names) & set(pathway_gene_list)
    # All available in CDT gene list
    available = set(cdt_genes) & set(pathway_gene_list)

    pathway_results.append({
        'pathway': pathway_name,
        'genes_in_cdt': len(available),
        'output_hits': len(output_overlap),
        'input_hits': len(input_overlap),
        'output_genes': sorted(output_overlap),
        'input_genes': sorted(input_overlap),
    })

    print(f'\n--- {pathway_name} ---')
    print(f'  Pathway genes in CDT: {len(available)}')
    print(f'  In top 100 output (affected by TFRC KD): {len(output_overlap)}')
    if output_overlap:
        print(f'    Genes: {", ".join(sorted(output_overlap))}')
    print(f'  In top 50 input (regulatory influence): {len(input_overlap)}')
    if input_overlap:
        print(f'    Genes: {", ".join(sorted(input_overlap))}')

df_pathway = pd.DataFrame(pathway_results)
print('\n')
print('='*70)

In [ ]:
# ── Pathway enrichment bar chart ──
fig, ax = plt.subplots(figsize=(10, 5))

pathways = df_pathway['pathway']
x = np.arange(len(pathways))
width = 0.35

bars1 = ax.bar(x - width/2, df_pathway['output_hits'], width,
               label='Output genes (affected by TFRC KD)', color='#E74C3C', edgecolor='black', linewidth=0.5)
bars2 = ax.bar(x + width/2, df_pathway['input_hits'], width,
               label='Input genes (regulatory influence)', color='#2980B9', edgecolor='black', linewidth=0.5)

ax.set_ylabel('Number of genes', fontsize=11)
ax.set_title('TFRC Perturbation: Pathway Overlap with Jacobian-Predicted Genes\n'
             'Relevance to PPMX-T003 (anti-TfR1 antibody) adverse effects', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(pathways, rotation=30, ha='right', fontsize=9)
ax.legend(fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Add count labels
for bar in bars1:
    if bar.get_height() > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{int(bar.get_height())}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    if bar.get_height() > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{int(bar.get_height())}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_BASE / 'tfrc_pathway_enrichment.pdf', bbox_inches='tight')
plt.savefig(OUTPUT_BASE / 'tfrc_pathway_enrichment.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: tfrc_pathway_enrichment.pdf/png')

In [ ]:
# ── Clinical relevance: top TFRC-affected genes with gradient scores ──
# Rank all genes by gradient importance under TFRC perturbation
tfrc_ranked = pd.DataFrame({
    'gene': cdt_genes,
    'gradient_importance': grad_tfrc,
    'experimental_effect': exp_tfrc,
}).sort_values('gradient_importance', ascending=False)

# Flag pathway membership
all_pathway_genes = {}
for pw_name, pw_genes in pathway_genes.items():
    for g in pw_genes:
        if g not in all_pathway_genes:
            all_pathway_genes[g] = []
        all_pathway_genes[g].append(pw_name)

tfrc_ranked['pathways'] = tfrc_ranked['gene'].map(
    lambda g: '; '.join(all_pathway_genes.get(g, [])) if g in all_pathway_genes else '')

print('='*70)
print('CLINICAL INTERPRETATION: TFRC Perturbation → PPMX-T003 Side Effects')
print('='*70)

print('\n--- Top 30 genes most affected by TFRC knockdown (by gradient) ---')
print(f'{"Rank":>4} {"Gene":<12} {"Gradient":>10} {"Exp Effect":>10}  Pathways')
print('-'*70)
for rank, (_, row) in enumerate(tfrc_ranked.head(30).iterrows(), 1):
    pw = row['pathways'] if row['pathways'] else '-'
    print(f'{rank:4d} {row["gene"]:<12} {row["gradient_importance"]:10.6f} '
          f'{row["experimental_effect"]:10.6f}  {pw}')

# --- Clinical connection summary ---
print('\n' + '='*70)
print('PPMX-T003 SIDE EFFECT PREDICTION SUMMARY')
print('='*70)
print('''
PPMX-T003 Clinical Findings (Phase 1):
  1. Reticulocyte decrease >50% (day 3)
  2. Hematocrit/hemoglobin decrease (day 7)
  3. Microcytic anemia tendency (PV patients)
  4. Infusion-related reactions

CDT-II Jacobian Predictions for TFRC KD:
  - Check output above for pathway overlaps
  - Iron metabolism genes in top affected → explains anemia
  - Ferroptosis genes in top affected → explains anti-tumor mechanism
  - Hematopoiesis genes affected → explains reticulocyte decrease
  - LAT1 axis genes → consistent with Leukemia 2024 paper
    (amino acid influx via LAT1 regulates iron demand)

Implication: CDT-II can computationally predict which biological
pathways will be disrupted by a drug target perturbation, enabling
side effect anticipation BEFORE clinical trials.
''')

# Save results
tfrc_ranked.to_csv(OUTPUT_BASE / 'tfrc_gradient_ranked_with_pathways.csv', index=False)
print(f'Saved: tfrc_gradient_ranked_with_pathways.csv')

## Summary

In [ ]:
print('='*60)
print('JACOBIAN HEATMAP & PATHWAY ANALYSIS RESULTS')
print('='*60)
print(f'\nGFI1B: Pearson r = {r_gfi1b:.4f}')
print(f'TFRC:  Pearson r = {r_tfrc:.4f}')
print(f'\nOutputs saved to: {OUTPUT_BASE}')
print(f'  - jacobian_heatmap_GFI1B.pdf/png')
print(f'  - jacobian_heatmap_TFRC.pdf/png')
print(f'  - tfrc_pathway_enrichment.pdf/png')
print(f'  - tfrc_gradient_ranked_with_pathways.csv')
print('='*60)